# GeoTransformer for Faces — Model Walkthrough

**Structure:**
1. **Full end-to-end prediction** — load a sample, run `model.forward()`, inspect every output at once
2. **Step-by-step dissection** — re-run each sub-module manually with visualisations

**Pipeline summary:**
```
src (raw)  ──► CrossAttentionRegressor ──► pred_scale, z_delta [32,100]
                                               |              |
                                          src/pred_scale   PCA morph ──► morphed_ref
                                               |              |
                                        cat(morphed_ref, scaled_src)
                                               |
                                        KPConv 4-stage graph (dynamic)
                                               |
                                        KPConvFPN ──► feats_f, feats_c
                                               |
                                        GeometricTransformer ──► norm feats
                                               |
                                        SuperPointMatching ──► coarse corr
                                               |
                                        Optimal Transport ──► fine scores
                                               |
                                        LocalGlobalRegistration ──► transform
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn.functional as F
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pytorch3d.ops import sample_farthest_points, knn_points

from geotransformer.datasets.registration.threedmatch.dataset import ThreeDMatchPairDataset
from geotransformer.utils.data import registration_collate_fn_stack_mode, precompute_data_stack_mode
from geotransformer.utils.torch import to_cuda
from geotransformer.utils.registration import compute_registration_error
from geotransformer.modules.ops import point_to_node_partition, index_select

from config_dowsampled import make_cfg
from model import create_model

print('Imports OK')

In [ ]:
def pts_trace(pts, color, name, size=2, opacity=1.0, symbol='circle'):
    if isinstance(pts, torch.Tensor):
        pts = pts.detach().cpu().numpy()
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity, symbol=symbol),
        name=name,
    )

def lines_trace(pts_a, pts_b, color='blue', name='correspondences', width=1):
    if isinstance(pts_a, torch.Tensor): pts_a = pts_a.detach().cpu().numpy()
    if isinstance(pts_b, torch.Tensor): pts_b = pts_b.detach().cpu().numpy()
    xs, ys, zs = [], [], []
    for a, b in zip(pts_a, pts_b):
        xs += [a[0], b[0], None]
        ys += [a[1], b[1], None]
        zs += [a[2], b[2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=width), name=name)

def show(*traces, title='', height=600):
    fig = go.Figure(data=list(traces))
    fig.update_layout(title=title, height=height,
                      scene=dict(aspectmode='data'),
                      margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

def rgb_colors(rgb_arr):
    return [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b in rgb_arr]

print('Helpers OK')

In [ ]:
DEVICE      = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DATASET_IDX = 10
WEIGHTS     = '../../output/geotransformer.facesdownsampledpointnetpp.stage4.gse.k3.max.oacl.stage2.sinkhorn/snapshots/epoch-40.pth.tar'
print(f'Device: {DEVICE}')

In [ ]:
cfg   = make_cfg()
model = create_model(cfg).to(DEVICE)
state = torch.load(WEIGHTS, map_location=DEVICE)
model.load_state_dict(state['model'])
model.eval()
model.neighbor_limits = [38, 36, 36, 38]

param_count = sum(p.numel() for p in model.parameters())
print(f'Model loaded  ({param_count:,} parameters)')
print(f'  pca_basis    : {model.pca_basis.shape}')
print(f'  pca_mean     : {model.pca_mean.shape}')
print(f'  patch_indices: {model.patch_indices.shape}')

---
# Part 1 — Full End-to-End Prediction

Load one validation sample, run `model.forward()` as a black box,
then inspect every output tensor before dissecting the internals in Part 2.

In [ ]:
planck_points = np.load('./plank.npy').astype(np.float32)

In [ ]:
pablo_points = np.load('./2189.npy').astype(np.float32)

In [ ]:
dataset = ThreeDMatchPairDataset('../../data/faces', 'val', use_augmentation=False)
sample  = dataset[DATASET_IDX]

sample["src_points"] = pablo_points

print(f'Sample keys : {list(sample.keys())}')
print(f'ref_points  : {sample["ref_points"].shape}   full-mesh reference face')
print(f'src_points  : {sample["src_points"].shape}   partial source scan')
print(f'gt_z        : {sample["gt_z"].shape}    ground-truth PCA coefficients')
print(f'gt_scale    : {sample["gt_scale"]:.4f}         ground-truth scale')
print(f'transform   : {sample["transform"].shape}')

In [ ]:
src_feats = np.ones_like(sample["src_points"][:, :1])
sample["src_feats"] = src_feats

In [ ]:
show(
    pts_trace(sample['ref_points'], 'red',   f'Reference  ({sample["ref_points"].shape[0]} pts)'),
    pts_trace(sample['src_points'], 'green', f'Source     ({sample["src_points"].shape[0]} pts)'),
    title='Input: Reference (red) vs Source (green) — unregistered'
)

In [ ]:
planck_points.dtype

In [ ]:
sample['src_feats'].shape

In [ ]:
# Build data_dict exactly as the dataloader does (precompute_data=False keeps points/lengths flat)
data_dict = registration_collate_fn_stack_mode(
    [sample],
    cfg.backbone.num_stages,
    cfg.backbone.init_voxel_size,
    cfg.backbone.init_radius,
    model.neighbor_limits,
    precompute_data=False,
)
data_dict = to_cuda(data_dict)

print('Running model.forward() ...')
with torch.no_grad():
    output_dict = model(data_dict)
print('Done.')

In [ ]:
print('=== output_dict tensors ===')
for k, v in output_dict.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:<38s} {str(v.shape):<25s} {v.dtype}')
    else:
        print(f'  {k:<38s} {type(v).__name__}')

In [ ]:
gt_T = torch.from_numpy(sample['transform']).float()
rre, rte = compute_registration_error(gt_T, output_dict['estimated_transform'].cpu())

print('=== Key predictions ===')
print(f'  pred_scale            : {output_dict["pred_scale"].item():.4f}   (gt = {sample["gt_scale"]:.4f})')
print(f'  scale error           : {abs(output_dict["pred_scale"].item() - sample["gt_scale"]):.4f}')
print(f'  superpoint pairs      : {output_dict["ref_node_corr_indices"].shape[0]}')
print(f'  final correspondences : {output_dict["ref_corr_points"].shape[0]}')
print(f'  RRE                   : {rre:.4f} deg')
print(f'  RTE                   : {rte:.6f}')
print()
print('Estimated transform:')
print(output_dict['estimated_transform'].cpu().numpy())
print('\nGround-truth transform:')
print(gt_T.numpy())

In [ ]:
# Morphed ref (from predicted z_delta) vs GT reconstruction vs input ref
show(
    pts_trace(sample['ref_points'],          'gray',      'Input ref (dataset)',          size=1, opacity=0.5),
    pts_trace(output_dict['morphed_full'],   'royalblue', 'Morphed ref (pred z_delta)',   size=2),
    pts_trace(output_dict['recon_gt_points'],'orange',    'Reconstructed ref (GT z)',     size=2, opacity=0.7),
    title='Morphed reference: predicted (blue) vs GT (orange) vs input (gray)'
)

In [ ]:
R_e = output_dict['estimated_transform'][:3, :3]
t_e = output_dict['estimated_transform'][:3,  3]
R_g = gt_T.to(DEVICE)[:3, :3]
t_g = gt_T.to(DEVICE)[:3,  3]

src_est = (output_dict['src_points'] @ R_e.T) + t_e
src_gt  = (output_dict['src_points'] @ R_g.T) + t_g

show(
    pts_trace(output_dict['ref_points'], 'red',       'Reference',               size=1),
    pts_trace(output_dict['src_points'], 'black',    'Source (unregistered)',    size=2, opacity=0.25),
    pts_trace(src_est,                   'limegreen', 'Predicted alignment',      size=2),
    pts_trace(src_gt,                    'royalblue', 'GT alignment',             size=1, opacity=0.5),
    title=f'Registration result   RRE={rre:.3f} deg   RTE={rte:.5f}'
)

In [ ]:
# Coarse correspondences overview
ref_c = output_dict['ref_points_c'][output_dict['ref_node_corr_indices']]
src_c = output_dict['src_points_c'][output_dict['src_node_corr_indices']]
show(
    pts_trace(output_dict['ref_points_c'], 'red',   f'Ref superpoints ({output_dict["ref_points_c"].shape[0]})', size=4),
    pts_trace(output_dict['src_points_c'], 'green', f'Src superpoints ({output_dict["src_points_c"].shape[0]})', size=4),
    lines_trace(ref_c, src_c, color='blue', name=f'{ref_c.shape[0]} coarse corr'),
    title='Coarse superpoint correspondences'
)

In [ ]:
# Fine correspondences used in the final SVD
show(
    pts_trace(output_dict['ref_points'], 'red',   'Reference', size=1, opacity=0.3),
    pts_trace(output_dict['src_points'], 'green', 'Source',    size=1, opacity=0.3),
    lines_trace(output_dict['ref_corr_points'], output_dict['src_corr_points'],
                color='blue', name=f'{output_dict["ref_corr_points"].shape[0]} final corr', width=2),
    title='Fine point correspondences used in weighted SVD'
)

---
# Part 2 — Step-by-Step Dissection

Re-run each sub-module individually, reusing tensors from `output_dict` where possible.

---
## Step 2 — Scale & Coefficient Prediction (`CrossAttentionRegressor`)

Takes raw source points and predicts:
- `pred_scale` in [0.4, 1.6] — size ratio src vs canonical reference  
- `z_delta` [32, 100] — per-patch PCA coefficients

Internals: FPS-1024 → center → **PointNet++ encoder** (FPS+kNN+MLP+maxpool → [B,128,256])
→ **cross-attention decoder** (32 patch tokens attend to memory) → linear [256→101]

In [ ]:
src_raw = torch.from_numpy(sample['src_points']).float().to(DEVICE)

num_samples = 1024
if src_raw.shape[0] > num_samples:
    src_fps, _ = sample_farthest_points(src_raw.unsqueeze(0), K=num_samples)
else:
    src_fps = src_raw.unsqueeze(0)

centroid     = src_fps.mean(dim=1, keepdim=True)
src_centered = src_fps - centroid
pad_mask     = torch.zeros((1, src_fps.shape[1]), dtype=torch.bool, device=DEVICE)

with torch.no_grad():
    z_delta_b, pred_scale_b = model.coeff_regressor(src_centered, pad_mask)

z_delta    = z_delta_b.squeeze(0)    # [32, 100]
pred_scale = pred_scale_b.squeeze()  # scalar

print(f'pred_scale : {pred_scale.item():.4f}   (gt = {sample["gt_scale"]:.4f})')
print(f'z_delta    : {z_delta.shape}  mean={z_delta.mean():.3f}  std={z_delta.std():.3f}')

In [ ]:
show(
    pts_trace(src_raw,              'lightgreen', f'Source full ({src_raw.shape[0]} pts)',    size=2, opacity=0.4),
    pts_trace(src_fps.squeeze(0),   'green',      f'FPS-{num_samples} (regressor input)',     size=4),
    pts_trace(src_centered.squeeze(0), 'royalblue','Centered',                               size=4),
    title='Step 2 — FPS downsampling and centering'
)

In [ ]:
# PointNet++ encoder: inspect one centroid neighbourhood
enc = model.coeff_regressor.point_encoder
with torch.no_grad():
    enc_cents, _ = sample_farthest_points(src_centered, K=enc.num_sampled_points)
    knn_res = knn_points(enc_cents, src_centered, K=enc.k_neighbors)

C = 30
nbr_idx = knn_res.idx[0, C].cpu().numpy()
show(
    pts_trace(src_centered.squeeze(0), 'lightblue', 'Centered src',                        size=2, opacity=0.6),
    pts_trace(enc_cents.squeeze(0),    'orange',    f'{enc.num_sampled_points} FPS centroids', size=4),
    pts_trace(src_centered[0, nbr_idx], 'green',      f'kNN-{enc.k_neighbors} of centroid {C}', size=1),
    pts_trace(enc_cents[0, C:C+1],     'black',     f'Centroid {C}',                       size=6),
    title=f'Step 2 — PointNet++ kNN neighbourhood (centroid {C})'
)

In [ ]:
gt_z = sample['gt_z'].float()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Predicted z_delta', 'GT z'])
fig.add_trace(go.Heatmap(z=z_delta.detach().cpu().numpy(),
                         colorscale='RdBu', zmid=0, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=gt_z.numpy(),
                         colorscale='RdBu', zmid=0, showscale=True),  row=1, col=2)
fig.update_layout(title='Step 2 — Predicted vs GT PCA coefficients  [32 patches x 100 components]',
                  height=420)
fig.show()
print(f'Coefficient MSE: {((z_delta.cpu() - gt_z)**2).mean().item():.6f}')

---
## Step 3 — PCA Morphing (`generate_reference_geometry`)

```
patch[p] = pca_mean[p] + z_delta[p] @ pca_basis[p]   # [K*3]
```
32 patches are stitched back to the full mesh with **last-write-wins** for overlapping vertices.

In [ ]:
with torch.no_grad():
    morphed_pred = model.generate_reference_geometry(z_delta)
    morphed_gt   = model.generate_reference_geometry(gt_z.to(DEVICE))
    morphed_zero = model.generate_reference_geometry(torch.zeros_like(z_delta))

disp = (morphed_pred - morphed_gt).norm(dim=1)
print(f'morphed_pred: {morphed_pred.shape}')
print(f'Displacement pred vs GT: mean={disp.mean():.5f}  max={disp.max():.5f}')

In [ ]:
import plotly.colors as pc
palette = pc.qualitative.Alphabet

num_patches = model.patch_indices.shape[0]
K = model.patch_indices.shape[1]
with torch.no_grad():
    delta_pts = torch.matmul(z_delta.unsqueeze(1), model.pca_basis).squeeze(1)
    patch_pts = (model.pca_mean + delta_pts).view(num_patches, K, 3)

traces = [pts_trace(patch_pts[p].cpu().numpy(), palette[p % len(palette)], f'Patch {p}', size=2)
          for p in range(num_patches)]
fig = go.Figure(data=traces)
fig.update_layout(title='Step 3 — 32 PCA patches before stitching',
                  scene=dict(aspectmode='data'), margin=dict(l=0,r=0,t=40,b=0))
fig.show()

In [ ]:
show(
    pts_trace(morphed_zero, 'gray',       'Template (z=0)',       size=1, opacity=0.5),
    pts_trace(morphed_pred, 'royalblue',  'Morphed pred',         size=2),
    pts_trace(morphed_gt,   'orange',     'Morphed GT',           size=2, opacity=0.7),
    pts_trace(sample['ref_points'], 'red','Dataset ref',          size=1, opacity=0.4),
    title='Step 3 — Template vs Morphed-pred vs Morphed-GT vs Input ref'
)

In [ ]:
disp_np = disp.detach().cpu().numpy()
mp_np   = morphed_pred.detach().cpu().numpy()
fig = go.Figure(go.Scatter3d(
    x=mp_np[:,0], y=mp_np[:,1], z=mp_np[:,2], mode='markers',
    marker=dict(size=2, color=disp_np, colorscale='Plasma', colorbar=dict(title='|pred-GT|')),
))
fig.update_layout(title='Step 3 — Per-vertex displacement  |morphed_pred - morphed_GT|',
                  scene=dict(aspectmode='data'), margin=dict(l=0,r=0,t=40,b=0))
fig.show()

---
## Step 4 — Source Scaling

`src_scaled = src_raw / pred_scale`

In [ ]:
src_scaled = src_raw / pred_scale.detach()
print(f'pred_scale = {pred_scale.item():.4f}   (gt = {sample["gt_scale"]:.4f})')
print(f'src range before: {src_raw.min():.3f} to {src_raw.max():.3f}')
print(f'src range after : {src_scaled.min():.3f} to {src_scaled.max():.3f}')
print(f'morphed_ref     : {morphed_pred.min():.3f} to {morphed_pred.max():.3f}')

In [ ]:
show(
    pts_trace(morphed_pred, 'royalblue', 'Morphed ref',                   size=2, opacity=0.5),
    pts_trace(src_raw,      'orange',    'Source (original scale)',        size=3, opacity=0.5),
    pts_trace(src_scaled,   'limegreen', f'Source / {pred_scale.item():.3f}', size=3),
    title=f'Step 4 — Source scaling  (pred_scale = {pred_scale.item():.4f})'
)

---
## Step 5 — Dynamic KPConv Graph

4-stage voxel-downsampling + radius-neighbourhood graph built on `cat(morphed_ref, src_scaled)`.
Each stage doubles the voxel/radius size.

In [ ]:
new_points   = torch.cat([morphed_pred, src_scaled], dim=0)
flat_lengths = torch.tensor([morphed_pred.shape[0], src_scaled.shape[0]], dtype=torch.long)

with torch.no_grad():
    graph_dict = precompute_data_stack_mode(
        new_points.cpu(), flat_lengths.cpu(),
        model.num_stages, model.init_voxel_size,
        model.init_radius, model.neighbor_limits,
    )
    for key in ['points','lengths','neighbors','subsampling','upsampling']:
        if key in graph_dict:
            graph_dict[key] = [t.to(DEVICE) for t in graph_dict[key]]

for s in range(len(graph_dict['points'])):
    pts  = graph_dict['points'][s]
    lens = graph_dict['lengths'][s]
    print(f'  Stage {s}: total={pts.shape[0]:6d}  ref={lens[0].item():6d}  src={lens[1].item():6d}'
          f'  voxel~{model.init_voxel_size * 2**s:.4f}')

In [ ]:
colors_ref = ['red', 'tomato', 'salmon', 'mistyrose']
colors_src = ['green', 'limegreen', 'lightgreen', 'honeydew']
traces = []
for s in range(len(graph_dict['points'])):
    pts  = graph_dict['points'][s]
    lens = graph_dict['lengths'][s]
    r, sv = pts[:lens[0].item()], pts[lens[0].item():]
    sz = max(1, 5 - s)
    traces += [
        pts_trace(r,  colors_ref[s], f'Ref stage {s} ({r.shape[0]} pts)',  size=sz),
        pts_trace(sv, colors_src[s], f'Src stage {s} ({sv.shape[0]} pts)', size=sz),
    ]
fig = go.Figure(data=traces)
fig.update_layout(title='Step 5 — All 4 KPConv graph stages (darker = finer)',
                  scene=dict(aspectmode='data'), margin=dict(l=0,r=0,t=40,b=0))
fig.show()

---
## Step 6 — KPConvFPN Backbone

Kernel Point Convolution FPN extracts multi-scale geometric features.
- `feats_f` (stage 0, fine) → patch matching
- `feats_c` (stage 3, coarse) → Geometric Transformer

In [ ]:
features_in = torch.ones(new_points.shape[0], 1, device=DEVICE)
bb_data = dict(graph_dict)
bb_data['features'] = features_in

with torch.no_grad():
    feats_list = model.backbone(features_in, bb_data)

feats_f = feats_list[0]
feats_c = feats_list[-1]
print(f'feats_f (fine,   stage 0): {feats_f.shape}')
print(f'feats_c (coarse, stage 3): {feats_c.shape}')

In [ ]:
from sklearn.decomposition import PCA as SklearnPCA

lens_c    = graph_dict['lengths'][-1]
ref_len_c = lens_c[0].item()
pts_c_np  = graph_dict['points'][-1].detach().cpu().numpy()

pca3 = SklearnPCA(n_components=3)
f3d  = pca3.fit_transform(feats_c.detach().cpu().numpy())
frgb = (f3d - f3d.min(0)) / (f3d.max(0) - f3d.min(0) + 1e-8)

fig = go.Figure([
    go.Scatter3d(x=pts_c_np[:ref_len_c,0], y=pts_c_np[:ref_len_c,1], z=pts_c_np[:ref_len_c,2],
                 mode='markers', marker=dict(size=5, color=rgb_colors(frgb[:ref_len_c])),
                 name='Ref coarse (feat PCA->RGB)'),
    go.Scatter3d(x=pts_c_np[ref_len_c:,0], y=pts_c_np[ref_len_c:,1], z=pts_c_np[ref_len_c:,2],
                 mode='markers', marker=dict(size=5, color=rgb_colors(frgb[ref_len_c:]), symbol='diamond'),
                 name='Src coarse (feat PCA->RGB)'),
])
fig.update_layout(title='Step 6 — Backbone coarse features (PCA->RGB)  matching colours = similar geometry',
                  scene=dict(aspectmode='data'), margin=dict(l=0,r=0,t=40,b=0))
fig.show()

---
## Step 7 — Geometric Transformer

Alternating `[self, cross] x 3` attention with geometric embeddings (distance + angles).
Output features are L2-normalised.

In [ ]:
# Reuse from output_dict (already computed by forward)
ref_feats_c_norm = output_dict['ref_feats_c']   # [N_ref_c, 256]
src_feats_c_norm = output_dict['src_feats_c']   # [N_src_c, 256]
ref_pts_c = output_dict['ref_points_c']         # [N_ref_c, 3]
src_pts_c = output_dict['src_points_c']         # [N_src_c, 3]

print(f'ref superpoints: {ref_pts_c.shape[0]}  feats: {ref_feats_c_norm.shape}')
print(f'src superpoints: {src_pts_c.shape[0]}  feats: {src_feats_c_norm.shape}')

In [ ]:
sim = (ref_feats_c_norm @ src_feats_c_norm.T).detach().cpu().numpy()
fig = px.imshow(sim, aspect='auto',
                labels=dict(x='Src superpoint', y='Ref superpoint', color='Cosine sim'),
                title='Step 7 — Transformer feature similarity matrix',
                color_continuous_scale='Viridis')
fig.show()
print(f'Similarity: mean={sim.mean():.3f}  max={sim.max():.3f}  min={sim.min():.3f}')

In [ ]:
all_tf = torch.cat([ref_feats_c_norm, src_feats_c_norm], 0).detach().cpu().numpy()
pca3t  = SklearnPCA(n_components=3)
tf3d   = pca3t.fit_transform(all_tf)
tfrgb  = (tf3d - tf3d.min(0)) / (tf3d.max(0) - tf3d.min(0) + 1e-8)

nrsp = ref_pts_c.shape[0]
fig = go.Figure([
    go.Scatter3d(x=ref_pts_c[:,0].cpu(), y=ref_pts_c[:,1].cpu(), z=ref_pts_c[:,2].cpu(),
                 mode='markers', marker=dict(size=6, color=rgb_colors(tfrgb[:nrsp])),
                 name='Ref (post-transformer)'),
    go.Scatter3d(x=src_pts_c[:,0].cpu(), y=src_pts_c[:,1].cpu(), z=src_pts_c[:,2].cpu(),
                 mode='markers', marker=dict(size=6, color=rgb_colors(tfrgb[nrsp:]), symbol='diamond'),
                 name='Src (post-transformer)'),
])
fig.update_layout(title='Step 7 — Post-transformer features (PCA->RGB)  same colour = compatible patch',
                  scene=dict(aspectmode='data'), margin=dict(l=0,r=0,t=40,b=0))
fig.show()

---
## Step 8 — Coarse Superpoint Matching

`SuperPointMatching` picks top-K pairs using dual-normalised cosine similarity.

In [ ]:
ref_nc = output_dict['ref_node_corr_indices']   # [K]
src_nc = output_dict['src_node_corr_indices']   # [K]

pair_scores = (ref_feats_c_norm[ref_nc] * src_feats_c_norm[src_nc]).sum(dim=1).detach().cpu().numpy()
print(f'Superpoint correspondences: {ref_nc.shape[0]}')
print(f'Cosine sim: mean={pair_scores.mean():.3f}  min={pair_scores.min():.3f}  max={pair_scores.max():.3f}')

In [ ]:
show(
    pts_trace(ref_pts_c, 'red',   f'Ref superpoints ({ref_pts_c.shape[0]})', size=4),
    pts_trace(src_pts_c, 'green', f'Src superpoints ({src_pts_c.shape[0]})', size=4),
    lines_trace(ref_pts_c[ref_nc], src_pts_c[src_nc],
                color='blue', name=f'{ref_nc.shape[0]} matched pairs'),
    title='Step 8 — Coarse superpoint correspondences'
)

In [ ]:
fig = px.histogram(pair_scores, nbins=30,
                   title='Step 8 — Cosine similarity of matched superpoints',
                   labels={'value': 'Cosine sim', 'count': 'Pairs'})
fig.show()

---
## Step 9 — Fine Patch Matching (Optimal Transport)

For each matched superpoint pair: extract local patch -> dot-product -> Sinkhorn OT.
Dustbin row/col absorbs unmatched points.

In [ ]:
ref_knn_pts   = output_dict['ref_node_corr_knn_points']   # [K, patch_size, 3]
src_knn_pts   = output_dict['src_node_corr_knn_points']   # [K, patch_size, 3]
ref_knn_masks = output_dict['ref_node_corr_knn_masks']    # [K, patch_size]
src_knn_masks = output_dict['src_node_corr_knn_masks']    # [K, patch_size]
matching_scores = output_dict['matching_scores']           # [K, P+1, P+1]

print(f'Patch pairs   : {ref_knn_pts.shape[0]}')
print(f'Max patch size: {ref_knn_pts.shape[1]}')
print(f'OT matrix     : {matching_scores.shape}  (includes dustbin)')

In [ ]:
PAIR = 0
r_m = ref_knn_masks[PAIR].cpu()
s_m = src_knn_masks[PAIR].cpu()
r_p = ref_knn_pts[PAIR, r_m].detach().cpu().numpy()
s_p = src_knn_pts[PAIR, s_m].detach().cpu().numpy()
print(f'Patch {PAIR}: ref={r_p.shape[0]} pts  src={s_p.shape[0]} pts')
show(
    pts_trace(r_p, 'red',   f'Ref patch {PAIR}', size=5),
    pts_trace(s_p, 'green', f'Src patch {PAIR}', size=5),
    title=f'Step 9 — Patch pair {PAIR}'
)

In [ ]:
ot_sub = matching_scores[PAIR, :r_m.sum(), :s_m.sum()].detach().cpu().numpy()
fig = px.imshow(ot_sub, aspect='auto',
                labels=dict(x='Src pt', y='Ref pt', color='OT score'),
                title=f'Step 9 — OT score matrix for patch {PAIR}  (no dustbin)',
                color_continuous_scale='Viridis')
fig.show()

In [ ]:
scores_sub = matching_scores[PAIR, :r_m.sum(), :s_m.sum()]
best_src   = scores_sub.argmax(dim=1).cpu().numpy()
best_val   = scores_sub.max(dim=1).values.cpu().numpy()
keep       = best_val >= np.percentile(best_val, 50)

show(
    pts_trace(r_p, 'red',   'Ref patch', size=5),
    pts_trace(s_p, 'green', 'Src patch', size=5),
    lines_trace(r_p[keep], s_p[best_src[keep]],
                color='blue', name=f'Top-50% OT ({keep.sum()})'),
    title=f'Step 9 — Intra-patch fine correspondences (patch {PAIR})'
)

---
## Step 10 — Final Registration (Weighted SVD)

`LocalGlobalRegistration` aggregates all fine correspondences, weights them by
match score x superpoint score, and solves via weighted SVD with iterative refinement.

In [ ]:
ref_corr = output_dict['ref_corr_points']     # [M, 3]
src_corr = output_dict['src_corr_points']     # [M, 3]
corr_sc  = output_dict['corr_scores']         # [M]
T_est    = output_dict['estimated_transform'] # [4, 4]

print(f'Final correspondences: {ref_corr.shape[0]}')
print(f'Score range: {corr_sc.min():.4f} to {corr_sc.max():.4f}')
print(f'RRE: {rre:.4f} deg    RTE: {rte:.6f}')

In [ ]:
show(
    pts_trace(output_dict['ref_points'], 'red',   'Reference', size=1, opacity=0.3),
    pts_trace(output_dict['src_points'], 'green', 'Source',    size=1, opacity=0.3),
    lines_trace(ref_corr, src_corr, color='blue',
                name=f'{ref_corr.shape[0]} final corr', width=2),
    title='Step 10 — Final correspondences used for weighted SVD'
)

In [ ]:
R_e = T_est[:3,:3]; t_e = T_est[:3,3]
R_g = gt_T.to(DEVICE)[:3,:3]; t_g = gt_T.to(DEVICE)[:3,3]
src_est = (output_dict['src_points'] @ R_e.T) + t_e
src_gt  = (output_dict['src_points'] @ R_g.T) + t_g

show(
    pts_trace(output_dict['ref_points'], 'red',       'Reference',            size=2),
    pts_trace(output_dict['src_points'], 'orange',    'Source (orig)',         size=2, opacity=0.25),
    pts_trace(src_est,                   'limegreen', 'Predicted alignment',   size=2),
    pts_trace(src_gt,                    'royalblue', 'GT alignment',          size=2, opacity=0.5),
    title=f'Step 10 — Final alignment   RRE={rre:.3f} deg   RTE={rte:.5f}'
)

---
## Summary

| Step | Module | Key Output | Shape |
|------|--------|------------|-------|
| 2 | `CrossAttentionRegressor` | `pred_scale`, `z_delta` | scalar, [32,100] |
| 3 | `generate_reference_geometry` | `morphed_ref` | [~10k, 3] |
| 4 | — | `src_scaled = src / pred_scale` | [N_src, 3] |
| 5 | `precompute_data_stack_mode` | 4-stage KPConv graph | — |
| 6 | `KPConvFPN` | `feats_f`, `feats_c` | [N, 256] |
| 7 | `GeometricTransformer` | `ref/src_feats_c_norm` | [N_c, 256] |
| 8 | `SuperPointMatching` | `ref/src_node_corr_idx` | [256] |
| 9 | `LearnableLogOptimalTransport` | `matching_scores` | [K, P+1, P+1] |
| 10 | `LocalGlobalRegistration` | `estimated_transform` | [4, 4] |